# 第 13 章: 主成分分析の探索と可視化

累積寄与率と、第 1・第 2 主成分の散布図を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)

In [ ]:
@file:DependsOn("org.tribuo:tribuo-math:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter07.toMatrix
import chapter13.fitPca
import chapter13.standardizeBoston
import chapter13.topLoadings
import chapter13.transform
import java.io.File

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val bostonCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "Boston.csv")
// 散布図の色分けに CRIME を使うので、標準化の前のデータも残しておく
val raw = DataFrame.readCSV(bostonCsv)
val df = standardizeBoston(raw)
val x = df.toMatrix(df.columnNames())
val model = fitPca(x, nComponents = df.columnsCount())

## 主成分の数と累積寄与率

In [ ]:
val cumulative =
    dataFrameOf(
        "主成分の数" to (1..df.columnsCount()).toList(),
        "累積寄与率" to model.explainedVarianceRatio.runningReduce(Double::plus),
    )
cumulative.plot {
    line {
        x("主成分の数")
        y("累積寄与率")
    }
    points {
        x("主成分の数")
        y("累積寄与率")
    }
    hLine {
        yIntercept.constant(0.8)
    }
    layout.title = "主成分の数と累積寄与率"
}

In [ ]:
cumulative

## 第 1・第 2 主成分の散布図

In [ ]:
val projected = transform(model, x)
val scores =
    dataFrameOf(
        "PC1" to projected.rows.map { it[0] },
        "PC2" to projected.rows.map { it[1] },
        "CRIME" to raw["CRIME"].values().map { it.toString() },
    )
scores.plot {
    points {
        x("PC1")
        y("PC2")
        color("CRIME")
    }
    layout.title = "第 1・第 2 主成分で見た地区"
}

In [ ]:
scores.groupBy("CRIME").mean()

## 主成分への影響が大きい列

In [ ]:
(0 until 2).associate { i ->
    "第 ${i + 1} 主成分" to topLoadings(model.components.rows[i], df.columnNames(), k = 5)
}